> **Research note:** Notebook prose may describe earlier exploratory analyses. Use regenerated artifacts and the current README for the implemented methodology.

# Synthetic-news sentiment analysis

This is a diagnostic of synthetic text, not evidence of real-news alpha. Headlines were partially conditioned on same-day returns. We report per-ticker Pearson correlation between the prior trading day’s available sentiment and the current return.


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))
import numpy as np
import pandas as pd
from src.io.loaders import load_sentiment_features, load_log_returns
from configs.settings import DATA_PROC

sentiment = load_sentiment_features().copy()
returns = load_log_returns().sort_index()
sentiment['Date'] = pd.to_datetime(sentiment['Date'])
calendar = pd.DatetimeIndex(returns.index)
lagged = []
for ticker, group in sentiment.groupby('ticker'):
    scores = group.set_index('Date')['sentiment_score'].reindex(calendar).shift(1)
    lagged.append(pd.DataFrame({'Date': calendar, 'ticker': ticker,
                                'sentiment_score_lag1': scores.to_numpy()}))
lagged = pd.concat(lagged, ignore_index=True).dropna(subset=['sentiment_score_lag1'])
returns_long = returns.stack().rename('log_return').reset_index()
returns_long.columns = ['Date', 'ticker', 'log_return']
merged = lagged.merge(returns_long, on=['Date', 'ticker'], how='inner')
ic = merged.groupby('ticker').apply(lambda g: g['sentiment_score_lag1'].corr(g['log_return']))
ic = ic.dropna()
metrics = {'ic_lagged': float(ic.mean()), 'ic_per_ticker': {str(k): float(v) for k, v in ic.items()},
           'n_tickers': len(ic), 'sentiment_lag_days': 1, 'correlation_method': 'pearson',
           'news_source': 'synthetic; labels partly determined by same-day returns'}
with open(DATA_PROC / 'nlp_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Mean Pearson correlation, previous trading day:', metrics['ic_lagged'])
